# Normalized Time Comparison

Load one policy result store twice: once with raw simulator steps and once with activation-opportunities-per-agent time.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tasks.policies import load_policy_results

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "tasks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Edit this path to compare another run.
ZARR_PATH = PROJECT_ROOT / "artifacts/simulation_runs/run-20260513T155848.zarr"
ZARR_PATH

In [ ]:
raw = load_policy_results(ZARR_PATH, normalize_time=False)
normalized = load_policy_results(ZARR_PATH, normalize_time=True)

print("raw sizes:", dict(raw.sizes))
print("normalized sizes:", dict(normalized.sizes))
print("normalized attrs:")
for key in (
    "normalize_time",
    "time_axis",
    "raw_step_labels",
    "common_normalized_time_max",
):
    value = normalized.attrs.get(key)
    if key == "raw_step_labels" and isinstance(value, list):
        value = value[:5] + (["..."] if len(value) > 5 else [])
    print(f"  {key}: {value}")

In [ ]:
config_path = Path(raw.attrs["config_path"])
config = json.loads(config_path.read_text())
n_actors = int(config["simulator"]["actor_scheduler"]["n_actors"])
raw_steps = np.asarray(raw.coords["step"].values, dtype=float)

rows = []
for n_agents in raw.coords["n_agents"].values:
    n_agents = int(n_agents)
    effective_n_actors = min(n_actors, n_agents)
    time_valid = np.asarray(
        raw.coords["time_valid"].sel(n_agents=n_agents).values, dtype=bool
    )
    valid_raw_steps = raw_steps[time_valid]
    rows.append(
        pd.DataFrame(
            {
                "n_agents": n_agents,
                "raw_step": valid_raw_steps,
                "activation_opportunities_per_agent": valid_raw_steps
                * effective_n_actors
                / n_agents,
            }
        )
    )

time_table = pd.concat(rows, ignore_index=True)
display(
    time_table.pivot(
        index="raw_step",
        columns="n_agents",
        values="activation_opportunities_per_agent",
    ).head(12)
)
print("returned normalized step coordinate:")
print(np.asarray(normalized.coords["step"].values[:12]))

In [ ]:
preferred_observable = "centered_mean_welfare.mean"
observable_names = [str(value) for value in raw.coords["observable"].values]
observable = (
    preferred_observable
    if preferred_observable in observable_names
    else observable_names[0]
)


def mean_series(ds, n_agents):
    series = ds["x"].sel(n_agents=n_agents, observable=observable)
    reduce_dims = [dim for dim in series.dims if dim != "step"]
    return series.mean(dim=reduce_dims, skipna=True)


fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for n_agents in raw.coords["n_agents"].values:
    raw_series = mean_series(raw, n_agents).where(
        raw.coords["time_valid"].sel(n_agents=n_agents), drop=True
    )
    axes[0].plot(
        raw_series.coords["step"].values,
        raw_series.values,
        label=f"n={int(n_agents)}",
    )
    axes[1].plot(
        normalized.coords["step"].values,
        mean_series(normalized, n_agents).values,
        label=f"n={int(n_agents)}",
    )

axes[0].set_title(f"Raw simulator steps: {observable}")
axes[0].set_xlabel("simulator step")
axes[0].set_ylabel(observable)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title(f"Normalized time: {observable}")
axes[1].set_xlabel("Expected update opportunities per agent")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.show()

In [ ]:
endpoint_rows = []
for n_agents in raw.coords["n_agents"].values:
    last_valid_raw_step = raw.coords["last_valid_raw_step"].sel(
        n_agents=n_agents
    ).item()
    endpoint_rows.append(
        {
            "n_agents": int(n_agents),
            "raw_last_valid_value": float(
                mean_series(raw, n_agents).sel(step=last_valid_raw_step).values
            ),
            "normalized_common_endpoint_value": float(
                mean_series(normalized, n_agents).isel(step=-1).values
            ),
        }
    )

endpoint_values = pd.DataFrame(endpoint_rows).set_index("n_agents")
display(endpoint_values)

ax = endpoint_values.plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_title(f"Last valid raw sample vs normalized common endpoint: {observable}")
ax.set_ylabel(observable)
ax.grid(axis="y", alpha=0.3)
plt.show()